*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*

> This notebook contains the raw code for the Capstone Project. To understand the architectural decisions, the math behind the latent space, and the production *Gotchas* to avoid, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

## 1. The Anatomy of a Single Training Step
### Step 1: Establish a Reproducible Training Contract

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# A reproducible problem makes it easy to verify that training actually works.
# Synthetic targets derived from features ensure a consistent decision boundary.
torch.manual_seed(42)

features = torch.randn(1000, 4)
class_scores = torch.stack([
    features[:, 0] + features[:, 1],
    -features[:, 0] + features[:, 2],
    features[:, 3] - features[:, 1],
], dim=1)
targets = class_scores.argmax(dim=1)

train_dataset = TensorDataset(features[:800], targets[:800])
val_dataset = TensorDataset(features[800:], targets[800:])

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=use_cuda,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    pin_memory=use_cuda,
)

model = nn.Sequential(
    nn.Linear(4, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-2)

assert len(train_dataset) == 800
assert next(iter(train_loader))[0].shape == (64, 4)
assert model(torch.randn(2, 4, device=device)).shape == (2, 3)

### Step 2: Execute and Inspect One Batch

In [2]:
model.train()
inputs, targets = next(iter(train_loader))
inputs = inputs.to(device, non_blocking=use_cuda)
targets = targets.to(device, non_blocking=use_cuda)

# The standard optimization sequence: zero_grad -> forward -> backward -> step.
# Verify each operation explicitly before running a full epoch.
weight_before = model[0].weight.detach().clone()

optimizer.zero_grad(set_to_none=True)
outputs = model(inputs)
loss = criterion(outputs, targets)
loss.backward()

assert loss.ndim == 0
assert model[0].weight.grad is not None

optimizer.step()

assert not torch.equal(weight_before, model[0].weight)

### Step 3: Expand the Update into a Training Epoch

In [3]:
def train_one_epoch(model, loader, optimizer, criterion):
    # Each epoch processes every batch, computing gradients and updating parameters.
    # Accumulate loss across batches using loss.item() to avoid retaining graphs.
    model.train()
    running_loss = 0.0
    samples_seen = 0

    for inputs, targets in loader:
        inputs = inputs.to(
            device, non_blocking=use_cuda
        )
        targets = targets.to(
            device, non_blocking=use_cuda
        )

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        samples_seen += batch_size

    return running_loss / samples_seen

first_epoch_loss = train_one_epoch(
    model, train_loader, optimizer, criterion
)
assert isinstance(first_epoch_loss, float)
assert first_epoch_loss > 0.0

## 2. The Evaluation Phase & State Management
### Step 1: Separate Evaluation Behavior from Autograd

In [4]:
model.eval()
sample_inputs, _ = next(iter(val_loader))
sample_inputs = sample_inputs.to(device)

# Validation requires two settings: model.eval() for mode-dependent layers,
# and torch.inference_mode() to disable gradient recording for efficiency.
with torch.inference_mode():
    sample_outputs = model(sample_inputs)

assert model.training is False
assert sample_outputs.shape[1] == 3
assert not sample_outputs.requires_grad

### Step 2: Measure Loss and Accuracy

In [ ]:
def evaluate(model, loader, criterion):
    # Evaluation accumulates metrics without computing or applying gradients.
    # Use inference_mode() to avoid graph overhead.
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    samples_seen = 0

    with torch.inference_mode():
        for inputs, targets in loader:
            inputs = inputs.to(
                device, non_blocking=use_cuda
            )
            targets = targets.to(
                device, non_blocking=use_cuda
            )

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            correct_predictions += (
                outputs.argmax(dim=1) == targets
            ).sum().item()
            samples_seen += batch_size

    return {
        "loss": running_loss / samples_seen,
        "accuracy": correct_predictions / samples_seen,
    }

validation_metrics = evaluate(model, val_loader, criterion)
assert 0.0 <= validation_metrics["accuracy"] <= 1.0

## 3. Assembling the Complete Native Loop

In [10]:
epochs = 5

for epoch in range(epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    metrics = evaluate(model, val_loader, criterion)

    print(
        f"Epoch [{epoch + 1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {metrics['loss']:.4f} | "
        f"Val Accuracy: {metrics['accuracy']:.4f}"
    )

Epoch [1/5] | Train Loss: 0.4029 | Val Loss: 0.2530 | Val Accuracy: 0.9600
Epoch [2/5] | Train Loss: 0.2290 | Val Loss: 0.1560 | Val Accuracy: 0.9800
Epoch [3/5] | Train Loss: 0.1588 | Val Loss: 0.1176 | Val Accuracy: 0.9800
Epoch [4/5] | Train Loss: 0.1282 | Val Loss: 0.0978 | Val Accuracy: 0.9800
Epoch [5/5] | Train Loss: 0.1117 | Val Loss: 0.0910 | Val Accuracy: 0.9800


## 4. Gotchas & Reality Checks: Production Pitfalls
### Gotcha 1: Retaining Every Loss Graph

In [11]:
import torch

parameter = torch.tensor(2.0, requires_grad=True)

# Unsafe: every addition links another loss graph.
graph_total = torch.tensor(0.0)
for target in (5.0, 6.0, 7.0):
    loss = (parameter * 3.0 - target).pow(2)
    graph_total = graph_total + loss
assert graph_total.grad_fn is not None

# Safe: item() extracts a detached Python scalar.
scalar_total = 0.0
for target in (5.0, 6.0, 7.0):
    loss = (parameter * 3.0 - target).pow(2)
    scalar_total += loss.item()
assert isinstance(scalar_total, float)

### Gotcha 2: The `Permanent Eval` Bug


In [12]:
import torch
import torch.nn as nn

dropout = nn.Dropout(p=1.0)
features = torch.ones(8, requires_grad=True)

dropout.eval()
validation_output = dropout(features)

# Bug: a new training phase starts without train().
forgotten_output = dropout(features)
assert dropout.training is False
assert torch.equal(forgotten_output, features)

# Fix: restore training behavior explicitly.
dropout.train()
training_output = dropout(features)

assert dropout.training is True
assert torch.count_nonzero(training_output) == 0
assert torch.equal(validation_output, features)

### Gotcha 3: Accidental Gradient Accumulation

In [13]:
import torch

weight = torch.tensor([2.0], requires_grad=True)
inputs = torch.tensor([3.0])

(weight * inputs).sum().backward()
first_gradient = weight.grad.detach().clone()

# No reset: the next gradient is added to the first.
(weight * inputs).sum().backward()
accumulated_gradient = weight.grad.detach().clone()

assert torch.equal(
    accumulated_gradient, 2 * first_gradient
)

# Equivalent to zero_grad(set_to_none=True) for this tensor.
weight.grad = None
(weight * inputs).sum().backward()

assert torch.equal(weight.grad, first_gradient)